<a href="https://colab.research.google.com/github/rajeshsharma38/neural-network-learning/blob/main/MINST_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd

In [2]:
import torch.nn.functional as F

In [3]:
df = pd.read_csv('mnist_train.csv', header=None)
df

,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,784
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [36]:
df.shape
Ytr = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
Xtr = torch.tensor(df.iloc[:, 1:].values, dtype=torch.float32)
Xtr.shape, Ytr.shape, Xtr.dtype, Ytr.dtype

(torch.Size([60000, 784]), torch.Size([60000]), torch.float32, torch.int64)

In [37]:
#check kamming_normal initialisation
w1 = torch.randn(784, 100) / 784**0.5
b1 = torch.randn(100) * 0.01
w2 = torch.randn(100, 10) / 100**0.5
b2 = torch.randn(10) * 0

parameters = [w1, b1, w2, b2]

for p in parameters:
  p.requires_grad = True


In [38]:
print(sum(p.numel() for p in parameters))

79510


In [39]:
X_mean = Xtr.mean(0, keepdim=True)
X_std = Xtr.std(0, keepdim=True)
eps = 0.01

In [40]:
#forward pass
batch_size = 50
batch_norm = torch.nn.BatchNorm1d(num_features=100, momentum=0.01)
for i in range(40000): # Reduced iterations for faster debugging
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb = Xtr[ix]
  Yb = Ytr[ix]

  # Xb_mean = Xb.mean(0, keepdim=True)
  # Xb_std = Xb.std(0, keepdim=True)

  # X_mean = 0.998 * X_mean + 0.002 * Xb_mean
  # X_std = 0.998 * X_std + 0.002 * Xb_std

  # Xb = (Xb - Xb_mean) / (Xb_std + eps)

  hpreact = batch_norm(Xb @ w1 + b1)

  # Detach, convert to numpy, flatten, and then remove NaNs before plotting
  # import numpy as np # Import numpy if not already imported
  # hpreact_flat = hpreact.detach().numpy().flatten()
  # hpreact_flat = hpreact_flat[~np.isnan(hpreact_flat)] # Filter out NaN values

  # if hpreact_flat.size > 0: # Only plot if there's data after removing NaNs
  #   plt.hist(hpreact_flat, bins=200) # Plot a histogram of flattened hpreact values
  #   plt.title('Distribution of hpreact values (NaNs removed)') # Add a title
  #   plt.xlabel('Value') # Add x-axis label
  #   plt.ylabel('Frequency') # Add y-axis label
  #   plt.show() # Ensure the plot is displayed
  # else:
  #   print("No valid data to plot after removing NaNs from hpreact.")

  # break # The break statement is currently preventing the training loop from running

  h = torch.tanh(hpreact)

  logits = h @ w2 + b2
  loss = F.cross_entropy(logits, Yb)

  #print(loss.item())

  for p in parameters:
    p.grad = None

  loss.backward()

  lr = 0.1 if i < 20000 else 0.01

  for p in parameters:
    p.data += -lr * p.grad

  if i%100 == 0: # Print loss more frequently with fewer iterations
    print(loss.item())

2.280616521835327
0.3402049243450165
0.3037550747394562
0.3618411719799042
0.2230042964220047
0.3155680000782013
0.16360996663570404
0.370035320520401
0.10258394479751587
0.1642421931028366
0.20004913210868835
0.29247456789016724
0.1924876719713211
0.24924618005752563
0.11911715567111969
0.4342557489871979
0.23359358310699463
0.17981310188770294
0.2683662474155426
0.2574625313282013
0.15163733065128326
0.15070685744285583
0.12231352925300598
0.13843797147274017
0.08551366627216339
0.13396412134170532
0.2760845422744751
0.2560500204563141
0.2638018727302551
0.18533194065093994
0.1103777140378952
0.2122654765844345
0.12944331765174866
0.1704324334859848
0.10644122958183289
0.1464276909828186
0.15779410302639008
0.09642568230628967
0.16668815910816193
0.30967143177986145
0.27845561504364014
0.15356425940990448
0.16224391758441925
0.04773123562335968
0.11669707298278809
0.1251743733882904
0.12142766267061234
0.2464032620191574
0.15033374726772308
0.06679200381040573
0.25264838337898254
0.0

Now that you've trained the model, let's use it to predict digits on a small portion of the test set (`mnist_test.csv`). We'll take the first 10 examples to quickly see how it performs.

In [41]:
# Load the test dataset (mnist_test.csv)
df_test = pd.read_csv('mnist_test.csv', header=None)

# Take only the first 10 examples for prediction
X_test = torch.tensor(df_test.iloc[:, 1:].values, dtype=torch.float32)
Y_test = torch.tensor(df_test.iloc[:, 0].values, dtype=torch.long)

print(f"Test data shape (features): {X_test.shape}")
print(f"Test data shape (labels): {Y_test.shape}")

Test data shape (features): torch.Size([10000, 784])
Test data shape (labels): torch.Size([10000])


In [42]:
# Perform forward pass on the test data using the trained weights (w1, b1, w2, b2)
batch_norm.eval()
hpreact_test = X_test @ w1 + b1
h_test = torch.tanh(batch_norm(hpreact_test))
logits_test = h_test @ w2 + b2

# Get the predicted class by finding the index of the maximum logit value
predictions = torch.argmax(logits_test, dim=1)

match_count = 0

print("--- Predictions vs Actual Labels (First 10 Examples) ---")
for i in range(len(predictions)):
    if predictions[i].item() == Y_test[i].item():
      match_count += 1
print(f"Matched: {match_count} \n Unmatched: {len(predictions) - match_count}")
batch_norm.train()

--- Predictions vs Actual Labels (First 10 Examples) ---
Matched: 9722 
 Unmatched: 278


BatchNorm1d(100, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)

In [22]:
#X_tr1 = batch_norm(Xtr)

In [43]:
batch_norm.eval() # Set batch_norm to evaluation mode

hpreact_test = Xtr @ w1 + b1
h_test = torch.tanh(batch_norm(hpreact_test))
logits_test = h_test @ w2 + b2

# Get the predicted class by finding the index of the maximum logit value
predictions = torch.argmax(logits_test, dim=1)

match_count = 0

print("--- Predictions vs Actual Labels ---")
for i in range(len(predictions)):
    if predictions[i].item() == Ytr[i].item():
      match_count += 1
print(f"Matched: {match_count} \n Unmatched: {len(predictions) - match_count}")

batch_norm.train() # Set batch_norm back to training mode if you plan further training

--- Predictions vs Actual Labels ---
Matched: 59594 
 Unmatched: 406


BatchNorm1d(100, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)